# K-Means Clustering From Scratch

Implemented using **NumPy** — `sklearn` is only used to load the dataset.

In this notebook we implement **K-Means Clustering from scratch**, covering:
- What K-Means is and how it differs from supervised learning
- **Centroid Initialization** — how we start the algorithm
- **Assignment Step** — assigning each point to its nearest centroid
- **Update Step** — recomputing centroids from assigned points
- **Inertia** — measuring how tight the clusters are
- A clean, reusable `KMeans` class with `fit` and `predict`

Resources that helped me build this:

Video — Intuition & Implementation: https://youtu.be/5w5iUbTlpMQ?si=3wTOl3XJiYz4tICl

## What is K-Means Clustering?

Every algorithm in this repo so far has been **supervised** — we had labels and learned to predict them.

**K-Means is unsupervised** — there are no labels. The goal is to discover hidden structure in the data by grouping similar points together into **K clusters**.

### How it works

K-Means alternates between two steps until convergence:

1. **Assignment** — assign each point to the nearest centroid:
$$c^{(i)} = \arg\min_k \; \lVert x^{(i)} - \mu_k \rVert_2$$

2. **Update** — recompute each centroid as the mean of its assigned points:
$$\mu_k = \frac{1}{|C_k|} \sum_{i \in C_k} x^{(i)}$$

The algorithm converges when assignments stop changing (or a max iteration limit is reached).

### What it minimizes

K-Means minimizes **inertia** — the total within-cluster sum of squared distances:

$$J = \sum_{k=1}^{K} \sum_{i \in C_k} \lVert x^{(i)} - \mu_k \rVert_2^2$$

### The key parameter: K
You have to choose K in advance. A common heuristic is the **elbow method** — plot inertia vs K and look for where the curve bends.

Let's start coding 🤓.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs  # only used to generate synthetic clusters

## 2. Generate Data

We use `sklearn.datasets.make_blobs` to generate a synthetic dataset with well-separated clusters.
We deliberately hide the true labels from the model — K-Means will have to discover the clusters on its own.

In [ ]:
np.random.seed(42)

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

print(f"Dataset shape : {X.shape}")
print(f"True clusters : {np.unique(y_true)}")

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c="steelblue", alpha=0.6, edgecolors="k", linewidths=0.3)
plt.title("Raw Data — No Labels Shown")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 3. The KMeans Class

**`fit`** runs the full K-Means loop:
1. Initialize K centroids by randomly picking K training points
2. **Assignment step** — for each point, find the nearest centroid (vectorized with broadcasting)
3. **Update step** — recompute each centroid as the mean of its assigned points
4. Repeat until assignments don't change or `max_iters` is reached
5. Track **inertia** at every iteration so we can plot convergence

**`predict`** assigns new points to the nearest learned centroid.

Everything is vectorized — no Python loop over samples in the assignment step.

In [ ]:
class KMeans:
    def __init__(self, k=3, max_iters=100, random_state=None):
        self.k           = k
        self.max_iters   = max_iters
        self.random_state = random_state

    # ── Internals ───────────────────────────────────────────────────────────────

    def _init_centroids(self, X):
        """Pick K random points from X as initial centroids."""
        if self.random_state is not None:
            np.random.seed(self.random_state)
        indices = np.random.choice(len(X), self.k, replace=False)
        return X[indices].copy()

    def _assign(self, X, centroids):
        """
        Assign each point to its nearest centroid.
        Vectorized: computes all distances in one broadcast operation.

        X         : (m, n)
        centroids : (k, n)
        returns   : (m,)  cluster index for each point
        """
        # (m, 1, n) - (1, k, n) → (m, k, n) → (m, k)
        dists = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        return np.argmin(dists, axis=1)

    def _compute_inertia(self, X, labels, centroids):
        """Total within-cluster sum of squared distances."""
        return sum(
            np.sum((X[labels == k] - centroids[k]) ** 2)
            for k in range(self.k)
        )

    # ── Public API ──────────────────────────────────────────────────────────────

    def fit(self, X):
        """Run K-Means on feature matrix X."""
        self.centroids_    = self._init_centroids(X)
        self.inertia_history_ = []

        for _ in range(self.max_iters):
            labels = self._assign(X, self.centroids_)

            new_centroids = np.array([
                X[labels == k].mean(axis=0) if np.any(labels == k) else self.centroids_[k]
                for k in range(self.k)
            ])

            self.inertia_history_.append(self._compute_inertia(X, labels, new_centroids))

            if np.allclose(self.centroids_, new_centroids):
                break

            self.centroids_ = new_centroids

        self.labels_ = self._assign(X, self.centroids_)
        self.inertia_ = self.inertia_history_[-1]
        return self

    def predict(self, X):
        """Assign new points to the nearest learned centroid."""
        return self._assign(X, self.centroids_)

## 4. Fit the Model

We fit K-Means with K=4 (matching the true number of clusters) and inspect the result.

In [ ]:
km = KMeans(k=4, max_iters=100, random_state=42)
km.fit(X)

print(f"Converged inertia : {km.inertia_:.2f}")
print(f"Cluster sizes     : {np.bincount(km.labels_)}")

## 5. Visualize the Clusters

We plot the discovered cluster assignments alongside the learned centroids.

In [ ]:
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

plt.figure(figsize=(6, 5))
for k in range(km.k):
    mask = km.labels_ == k
    plt.scatter(X[mask, 0], X[mask, 1],
                color=COLORS[k], alpha=0.6, edgecolors="k", linewidths=0.3,
                label=f"Cluster {k}")

plt.scatter(km.centroids_[:, 0], km.centroids_[:, 1],
            c="black", marker="X", s=200, zorder=5, label="Centroids")

plt.title("K-Means Clustering (K=4)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 6. Convergence — Inertia Over Iterations

Inertia should decrease monotonically and flatten when the algorithm converges.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(km.inertia_history_) + 1), km.inertia_history_,
         marker="o", color="steelblue")
plt.xlabel("Iteration")
plt.ylabel("Inertia")
plt.title("K-Means Convergence")
plt.grid(True, alpha=0.3)
plt.show()

## 7. The Elbow Method — Choosing K

How do we pick K when we don't know the true number of clusters?

We fit K-Means for a range of K values and plot inertia. Inertia always decreases as K increases (more clusters = tighter fit), but the **rate of decrease slows** after the optimal K. The "elbow" in the curve is the sweet spot.

In [ ]:
k_range = range(1, 10)
inertias = []

for k in k_range:
    km_k = KMeans(k=k, max_iters=100, random_state=42)
    km_k.fit(X)
    inertias.append(km_k.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_range, inertias, marker="o", color="steelblue")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow Method — Choosing K")
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()